# NI-DAQ recorder controls

Start begins a new measurement and creates a new HDF5 file. Stop signals the worker threads, waits for them to finish, and closes (flushes) the file.


In [ ]:
import threading
import time
from datetime import datetime
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

import lib_ipmu_daq_api as api
# import lib_ipmu_daq_aquisition as acq
import lib_ipmu_daq_generator as gen
import lib_ipmu_daq_process as post

%load_ext autoreload
%autoreload 2


In [ ]:
class MeasurementController:
    def __init__(self, runs_dir=Path("./")):
        self.runs_dir = Path(runs_dir)
        self.app = None
        # self.acquisition_thread = None
        self.generator_thread = None
        self.processor_thread = None

        self.start_button = widgets.Button(
            description="Start", button_style="success", icon="play"
        )
        self.stop_button = widgets.Button(
            description="Stop", button_style="danger", icon="stop", disabled=True
        )
        self.status = widgets.HTML(value="Ready")
        self.start_button.on_click(self._start)
        self.stop_button.on_click(self._stop)

    def display(self):
        display(widgets.VBox([
            widgets.HBox([self.start_button, self.stop_button]),
            self.status,
        ]))

    def _start(self, _button=None):
        if self.app is not None:
            return

        self.start_button.disabled = True
        self.status.value = "Starting measurement..."
        try:
            self._wait_for_unused_filename()
            app = api.DAQApp(DEBUG=False)
            app.setup()
            app.initStorer(runs_dir=self.runs_dir)
            app.initLogger()

            # acquisition = acq.DataAquisition(
            #     buf_q=app.buf_q, stop_event=app.stop_event
            # )
            generator = gen.Generator(
                buf_q=app.buf_q, stop_event=app.stop_event
            )
            processor = post.Processor(
                buf_q=app.buf_q,
                quad_q=app.quad_q,
                comvel_q=app.comvel_q,
                h5f=app.h5f,
                dset=app.dset,
                stop_event=app.stop_event,
                logger=app.logger,
                debug=app.DEBUG,
            )

            self.app = app
            # self.acquisition_thread = threading.Thread(
            #     target=acquisition.run, daemon=True
            # )
            self.generator_thread = threading.Thread(
                target=generator.run, daemon=True
            )
            self.processor_thread = threading.Thread(
                target=processor.run, daemon=True
            )
            # self.acquisition_thread.start()
            self.generator_thread.start()
            self.processor_thread.start()

            self.stop_button.disabled = False
            self.status.value = f"Recording: <code>{app.h5f.filename}</code>"
        except Exception as exc:
            if self.app is not None:
                self.app.shutdown()
            self._clear_run()
            self.start_button.disabled = False
            self.status.value = f"Start failed: <code>{exc}</code>"
            raise

    def _stop(self, _button=None):
        if self.app is None:
            return

        self.stop_button.disabled = True
        self.status.value = "Stopping and flushing the HDF5 file..."
        app = self.app
        filename = app.h5f.filename
        app.shutdown()

        # self.acquisition_thread.join()
        self.generator_thread.join()
        self.processor_thread.join()
        self._clear_run()

        self.start_button.disabled = False
        self.status.value = f"Saved: <code>{filename}</code>"

    def _clear_run(self):
        self.app = None
        # self.acquisition_thread = None
        self.generator_thread = None
        self.processor_thread = None

    def _wait_for_unused_filename(self):
        # DAQApp filenames have one-second resolution. Avoid truncating a
        # file if Start is clicked again within the same second.
        self.runs_dir.mkdir(exist_ok=True)
        while (self.runs_dir / f"{datetime.now():%y%m%d%H%M%S}.h5").exists():
            time.sleep(0.05)


In [ ]:
controller = MeasurementController(runs_dir=Path("./"))
controller.display()
